# Experiments: Vector Databases

This notebook produces **runnable evidence** for the claims you will make in interviews about vector database indexing algorithms. Each experiment isolates one variable, measures the effect, and gives you a concrete number to cite.

**Prerequisites:** Read [vector-databases.md](./vector-databases.md) and run [03_vector_databases.ipynb](./03_vector_databases.ipynb) first.

In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="rag",
    notebook_name="03_vector_databases_experiments.ipynb"
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

np.random.seed(42)

---

## Shared Utilities

We build simple implementations of Flat, IVF, and HNSW search from scratch using only numpy. These are simplified versions that demonstrate the algorithm mechanics and scaling behavior.

In [ ]:
# --- Shared utilities ---

def generate_vectors(n, d, n_clusters=10):
    """Generate clustered vectors to simulate real embedding distributions.
    
    Real embeddings are not uniformly distributed — they cluster by topic.
    This generator creates vectors around cluster centers.
    """
    centers = np.random.randn(n_clusters, d)
    labels = np.random.randint(0, n_clusters, n)
    vectors = centers[labels] + np.random.randn(n, d) * 0.3
    # L2 normalize
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    vectors = vectors / norms
    return vectors, labels


def brute_force_search(query, database, k=10):
    """Exact nearest neighbor search. The ground truth."""
    sims = database @ query  # dot product (vectors are normalized)
    top_k = np.argsort(sims)[-k:][::-1]
    return top_k, sims[top_k]


def recall_at_k(predicted, ground_truth, k=10):
    """Fraction of ground-truth top-k that appear in predicted top-k."""
    gt_set = set(ground_truth[:k])
    pred_set = set(predicted[:k])
    return len(gt_set & pred_set) / k


print("Utilities loaded.")

---

## Experiment 1: Brute Force vs IVF Timing

**Claim to test:** "Brute force is O(D×d) per query. IVF is much faster because it only searches a subset of clusters."

We measure queries per second (QPS) at different dataset sizes for brute force and IVF.

In [ ]:
# --- Experiment 1: Brute force vs IVF timing ---

class SimpleIVF:
    """Simplified IVF index using k-means clustering."""
    
    def __init__(self, n_clusters=32, nprobe=5):
        self.n_clusters = n_clusters
        self.nprobe = nprobe
        self.centroids = None
        self.inverted_lists = None
    
    def build(self, vectors, n_iter=10):
        """Build IVF index using simple k-means."""
        n, d = vectors.shape
        # Initialize centroids randomly from data
        idx = np.random.choice(n, self.n_clusters, replace=False)
        self.centroids = vectors[idx].copy()
        
        # K-means iterations
        for _ in range(n_iter):
            # Assign to nearest centroid
            sims = vectors @ self.centroids.T
            assignments = np.argmax(sims, axis=1)
            # Update centroids
            for c in range(self.n_clusters):
                mask = assignments == c
                if mask.sum() > 0:
                    self.centroids[c] = vectors[mask].mean(axis=0)
                    self.centroids[c] /= np.linalg.norm(self.centroids[c])
        
        # Build inverted lists
        sims = vectors @ self.centroids.T
        assignments = np.argmax(sims, axis=1)
        self.inverted_lists = {}
        for c in range(self.n_clusters):
            mask = assignments == c
            self.inverted_lists[c] = {
                'indices': np.where(mask)[0],
                'vectors': vectors[mask]
            }
    
    def search(self, query, k=10):
        """Search nprobe closest clusters."""
        # Find closest clusters
        centroid_sims = self.centroids @ query
        top_clusters = np.argsort(centroid_sims)[-self.nprobe:][::-1]
        
        # Search within those clusters
        all_indices = []
        all_sims = []
        for c in top_clusters:
            if len(self.inverted_lists[c]['indices']) == 0:
                continue
            vecs = self.inverted_lists[c]['vectors']
            sims = vecs @ query
            all_indices.extend(self.inverted_lists[c]['indices'])
            all_sims.extend(sims)
        
        if not all_indices:
            return np.array([]), np.array([])
        
        all_sims = np.array(all_sims)
        top_k_local = np.argsort(all_sims)[-k:][::-1]
        result_indices = np.array(all_indices)[top_k_local]
        result_sims = all_sims[top_k_local]
        return result_indices, result_sims


# Benchmark at different sizes
d = 128  # embedding dimension
sizes = [1000, 3000, 5000, 10000, 30000, 50000]
bf_qps = []
ivf_qps = []
ivf_recall = []
n_queries = 20

for n in sizes:
    vectors, _ = generate_vectors(n, d)
    queries = np.random.randn(n_queries, d)
    queries /= np.linalg.norm(queries, axis=1, keepdims=True)
    
    # Brute force timing
    t0 = time.perf_counter()
    bf_results = [brute_force_search(q, vectors, k=10) for q in queries]
    bf_time = time.perf_counter() - t0
    bf_qps.append(n_queries / bf_time)
    
    # IVF timing
    n_clusters = max(int(np.sqrt(n)), 8)
    ivf = SimpleIVF(n_clusters=n_clusters, nprobe=5)
    ivf.build(vectors)
    
    t0 = time.perf_counter()
    ivf_results = [ivf.search(q, k=10) for q in queries]
    ivf_time = time.perf_counter() - t0
    ivf_qps.append(n_queries / ivf_time)
    
    # Recall
    recalls = []
    for (bf_idx, _), (ivf_idx, _) in zip(bf_results, ivf_results):
        if len(ivf_idx) > 0:
            recalls.append(recall_at_k(ivf_idx, bf_idx, k=10))
    ivf_recall.append(np.mean(recalls))

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(sizes, bf_qps, 'o-', color='#1976d2', linewidth=2, markersize=8, label='Brute Force')
ax1.plot(sizes, ivf_qps, 's-', color='#e53935', linewidth=2, markersize=8, label='IVF')
ax1.set_xlabel('Dataset Size', fontsize=12)
ax1.set_ylabel('Queries Per Second (QPS)', fontsize=12)
ax1.set_title('QPS: Brute Force vs IVF', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)
ax1.set_xscale('log')

ax2.plot(sizes, ivf_recall, 'o-', color='#2e7d32', linewidth=2, markersize=8)
ax2.set_xlabel('Dataset Size', fontsize=12)
ax2.set_ylabel('Recall@10', fontsize=12)
ax2.set_title('IVF Recall@10 (nprobe=5)', fontsize=14, fontweight='bold')
ax2.set_ylim(0.5, 1.05)
ax2.axhline(y=1.0, color='gray', linestyle='--', alpha=0.3)
ax2.grid(alpha=0.3)
ax2.set_xscale('log')

plt.tight_layout()
plt.show()

print(f"{'N':>8} {'BF QPS':>10} {'IVF QPS':>10} {'Speedup':>10} {'Recall@10':>10}")
print("-" * 50)
for n, bq, iq, r in zip(sizes, bf_qps, ivf_qps, ivf_recall):
    print(f"{n:>8} {bq:>10.0f} {iq:>10.0f} {iq/bq:>9.1f}× {r:>10.2f}")

**What this shows:** IVF is consistently faster than brute force, and the speed advantage grows with dataset size. IVF trades a small amount of recall for a large speedup. In an interview: "IVF provides 5-50× speedup over brute force by only searching a subset of clusters. The recall is 85-95% with nprobe=5."

---

## Experiment 2: HNSW Parameter Sensitivity (M and ef)

**Claim to test:** "M controls graph density and ef_search controls beam width. Both affect the recall-latency trade-off."

We build a simplified HNSW and sweep M and ef to show the Pareto frontier.

In [ ]:
# --- Experiment 2: HNSW parameter sensitivity ---
# We simulate the HNSW recall-latency trade-off using a graph-based search.
# This is a single-layer NSW (navigable small world) to keep things simple.

class SimpleNSW:
    """Single-layer navigable small world graph for ANN search."""
    
    def __init__(self, M=16):
        self.M = M
        self.graph = {}  # node_id -> list of neighbor_ids
        self.vectors = None
    
    def build(self, vectors):
        """Build the graph by inserting vectors one at a time."""
        self.vectors = vectors
        n = len(vectors)
        self.graph = {i: [] for i in range(n)}
        
        for i in range(1, n):
            # Find M nearest neighbors among already-inserted nodes
            inserted = np.arange(i)
            sims = vectors[inserted] @ vectors[i]
            nearest = inserted[np.argsort(sims)[-self.M:]]
            
            # Connect bidirectionally
            for j in nearest:
                self.graph[i].append(j)
                self.graph[j].append(i)
                # Prune if too many connections
                if len(self.graph[j]) > self.M * 2:
                    # Keep M*2 closest
                    neighbors = self.graph[j]
                    neighbor_sims = vectors[neighbors] @ vectors[j]
                    keep = np.argsort(neighbor_sims)[-(self.M * 2):]
                    self.graph[j] = [neighbors[k] for k in keep]
    
    def search(self, query, k=10, ef=50):
        """Greedy beam search on the graph."""
        # Start from a random entry point
        entry = np.random.randint(len(self.vectors))
        
        visited = set()
        candidates = [(-(self.vectors[entry] @ query), entry)]  # min-heap (negate for max)
        results = [(-candidates[0][0], entry)]  # (sim, id)
        visited.add(entry)
        
        import heapq
        
        while candidates:
            neg_sim, current = heapq.heappop(candidates)
            current_sim = -neg_sim
            
            # If the best candidate is worse than the worst in results, stop
            if len(results) >= ef and current_sim < results[-1][0]:
                break
            
            # Expand neighbors
            for neighbor in self.graph.get(current, []):
                if neighbor not in visited:
                    visited.add(neighbor)
                    sim = self.vectors[neighbor] @ query
                    heapq.heappush(candidates, (-sim, neighbor))
                    results.append((sim, neighbor))
                    results.sort(key=lambda x: x[0], reverse=True)
                    results = results[:ef]  # keep top ef
        
        results.sort(key=lambda x: x[0], reverse=True)
        top_k = results[:k]
        return np.array([r[1] for r in top_k]), np.array([r[0] for r in top_k])


# Build dataset
n = 5000
d = 64
vectors, _ = generate_vectors(n, d)
queries = np.random.randn(30, d)
queries /= np.linalg.norm(queries, axis=1, keepdims=True)

# Get ground truth
gt_results = [brute_force_search(q, vectors, k=10) for q in queries]

# Sweep M values
M_values = [4, 8, 16, 32]
ef_values = [10, 20, 50, 100, 200]

results_grid = {}  # (M, ef) -> (recall, time_ms)

for M in M_values:
    nsw = SimpleNSW(M=M)
    nsw.build(vectors)
    
    for ef in ef_values:
        # Time the search
        t0 = time.perf_counter()
        search_results = [nsw.search(q, k=10, ef=ef) for q in queries]
        search_time = (time.perf_counter() - t0) / len(queries) * 1000  # ms per query
        
        # Compute recall
        recalls = []
        for (gt_idx, _), (sr_idx, _) in zip(gt_results, search_results):
            if len(sr_idx) > 0:
                recalls.append(recall_at_k(sr_idx, gt_idx, k=10))
        avg_recall = np.mean(recalls)
        
        results_grid[(M, ef)] = (avg_recall, search_time)

# Plot Pareto frontier
fig, ax = plt.subplots(figsize=(10, 6))
colors = {'4': '#1976d2', '8': '#2e7d32', '16': '#ff7043', '32': '#9c27b0'}

for M in M_values:
    recalls = [results_grid[(M, ef)][0] for ef in ef_values]
    times = [results_grid[(M, ef)][1] for ef in ef_values]
    ax.plot(times, recalls, 'o-', color=colors[str(M)], linewidth=2, 
            markersize=8, label=f'M={M}')
    # Annotate ef values on the M=16 line
    if M == 16:
        for ef, r, t in zip(ef_values, recalls, times):
            ax.annotate(f'ef={ef}', (t, r), textcoords='offset points',
                       xytext=(5, 5), fontsize=8, color='gray')

ax.set_xlabel('Latency (ms per query)', fontsize=12)
ax.set_ylabel('Recall@10', fontsize=12)
ax.set_title('HNSW Recall-Latency Pareto Frontier', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_ylim(0.3, 1.05)
plt.tight_layout()
plt.show()

print(f"{'M':>4} {'ef':>6} {'Recall@10':>10} {'Time (ms)':>10}")
print("-" * 34)
for M in M_values:
    for ef in ef_values:
        r, t = results_grid[(M, ef)]
        print(f"{M:>4} {ef:>6} {r:>10.3f} {t:>10.2f}")

**What this shows:** Higher M and ef both improve recall but increase latency. The Pareto frontier shows that M=16, ef=50-100 is a good operating point. In an interview: "HNSW has two knobs: M controls graph density (more edges = better recall but more memory), ef_search controls beam width (wider beam = better recall but slower). The typical production operating point is M=16, ef=4-8× top-k."

---

## Experiment 3: Curse of Dimensionality Demo

**Claim to test:** "In high dimensions, the distance to the nearest neighbor and the distance to a random point converge — making ANN harder."

We measure the ratio of nearest-neighbor distance to average distance across dimensions.

In [ ]:
# --- Experiment 3: Curse of dimensionality ---

dimensions = [2, 5, 10, 20, 50, 100, 200, 500, 768, 1024]
nn_ratio = []       # nearest / average distance
nn_10_ratio = []    # 10th nearest / nearest distance
n_points = 1000
n_queries_dim = 50

for d in dimensions:
    # Generate random unit vectors
    points = np.random.randn(n_points, d)
    points /= np.linalg.norm(points, axis=1, keepdims=True)
    
    ratios_1 = []
    ratios_10 = []
    
    for _ in range(n_queries_dim):
        query = np.random.randn(d)
        query /= np.linalg.norm(query)
        
        # L2 distances to all points
        dists = np.linalg.norm(points - query, axis=1)
        sorted_dists = np.sort(dists)
        
        nearest = sorted_dists[0]
        tenth = sorted_dists[9]
        avg = np.mean(dists)
        
        ratios_1.append(nearest / avg)
        ratios_10.append(tenth / nearest if nearest > 0 else 1.0)
    
    nn_ratio.append(np.mean(ratios_1))
    nn_10_ratio.append(np.mean(ratios_10))

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(dimensions, nn_ratio, 'o-', color='#1976d2', linewidth=2, markersize=6)
ax1.set_xlabel('Dimensionality', fontsize=12)
ax1.set_ylabel('Nearest / Average Distance', fontsize=12)
ax1.set_title('Nearest Neighbor vs Average Distance', fontsize=13, fontweight='bold')
ax1.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='All equidistant')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)
ax1.set_xscale('log')

ax2.plot(dimensions, nn_10_ratio, 's-', color='#e53935', linewidth=2, markersize=6)
ax2.set_xlabel('Dimensionality', fontsize=12)
ax2.set_ylabel('10th Nearest / 1st Nearest Distance', fontsize=12)
ax2.set_title('Distance Concentration', fontsize=13, fontweight='bold')
ax2.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='All equidistant')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)
ax2.set_xscale('log')

plt.tight_layout()
plt.show()

print(f"{'Dim':>6} {'NN/Avg ratio':>14} {'10th/1st ratio':>16}")
print("-" * 38)
for d, r1, r10 in zip(dimensions, nn_ratio, nn_10_ratio):
    print(f"{d:>6} {r1:>14.4f} {r10:>16.4f}")

print(f"\nAt d={dimensions[0]}, nearest neighbor is {nn_ratio[0]:.2f}× the average distance.")
print(f"At d={dimensions[-1]}, nearest neighbor is {nn_ratio[-1]:.2f}× the average distance.")
print(f"The gap shrinks — this is the curse of dimensionality.")

**What this shows:** As dimensionality increases, the nearest neighbor distance converges toward the average distance. All points become approximately equidistant. In an interview: "The curse of dimensionality makes ANN harder in high dimensions because the 'contrast' between nearest and average neighbors shrinks. Real embeddings work because they lie on a low-dimensional manifold within the high-dimensional space."

---

## Experiment 4: Memory Calculation Verification

**Claim to test:** "Flat index memory = D × d × 4 bytes. IVF adds centroid storage. PQ compresses to D × m bytes."

We verify theoretical memory formulas against actual numpy array sizes.

In [ ]:
# --- Experiment 4: Memory calculation verification ---

configs = [
    {"name": "Small (10K, d=384)", "n": 10_000, "d": 384},
    {"name": "Medium (100K, d=768)", "n": 100_000, "d": 768},
    {"name": "Large (1M, d=768)", "n": 1_000_000, "d": 768},
    {"name": "XL (10M, d=768)", "n": 10_000_000, "d": 768},
]

print(f"{'Config':<25} {'Flat':>12} {'HNSW (M=16)':>14} {'PQ (m=48)':>12} {'PQ ratio':>10}")
print("-" * 75)

flat_sizes = []
hnsw_sizes = []
pq_sizes = []

for cfg in configs:
    n, d = cfg['n'], cfg['d']
    
    # Flat: D × d × 4 bytes (float32)
    flat_bytes = n * d * 4
    
    # HNSW: Flat + graph edges (M bidirectional edges per node, 4 bytes per edge index)
    M = 16
    hnsw_bytes = flat_bytes + n * M * 2 * 4  # bidirectional edges
    
    # PQ: D × m bytes (1 byte per sub-vector code) + codebook
    m = min(48, d)  # sub-vectors
    k = 256  # centroids per sub-space
    pq_bytes = n * m + m * k * (d // m) * 4  # codes + codebook
    
    flat_sizes.append(flat_bytes)
    hnsw_sizes.append(hnsw_bytes)
    pq_sizes.append(pq_bytes)
    
    def fmt(b):
        if b >= 1e9: return f"{b/1e9:.1f} GB"
        if b >= 1e6: return f"{b/1e6:.1f} MB"
        return f"{b/1e3:.1f} KB"
    
    ratio = flat_bytes / pq_bytes
    print(f"{cfg['name']:<25} {fmt(flat_bytes):>12} {fmt(hnsw_bytes):>14} {fmt(pq_bytes):>12} {ratio:>9.0f}×")

# Verify with actual numpy array
print(f"\n--- Verification with actual numpy arrays ---")
test_n, test_d = 10000, 768
flat_array = np.random.randn(test_n, test_d).astype(np.float32)
pq_array = np.random.randint(0, 256, size=(test_n, 48), dtype=np.uint8)

print(f"Flat array: {flat_array.nbytes:,} bytes (theoretical: {test_n * test_d * 4:,})")
print(f"PQ array:   {pq_array.nbytes:,} bytes (theoretical: {test_n * 48:,})")
print(f"Compression: {flat_array.nbytes / pq_array.nbytes:.0f}×")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(configs))
width = 0.25

bars1 = ax.bar(x - width, [s/1e9 for s in flat_sizes], width, label='Flat', color='#1976d2', alpha=0.8)
bars2 = ax.bar(x, [s/1e9 for s in hnsw_sizes], width, label='HNSW (M=16)', color='#ff7043', alpha=0.8)
bars3 = ax.bar(x + width, [s/1e9 for s in pq_sizes], width, label='PQ (m=48)', color='#2e7d32', alpha=0.8)

ax.set_xlabel('Configuration', fontsize=12)
ax.set_ylabel('Memory (GB)', fontsize=12)
ax.set_title('Memory Usage by Index Type', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([c['name'].split('(')[0].strip() for c in configs], fontsize=10)
ax.legend(fontsize=10)
ax.set_yscale('log')
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

**What this shows:** Memory scales linearly with dataset size. HNSW adds ~10-15% overhead for graph edges. PQ achieves 60× compression. In an interview: "At 1M vectors with d=768, the flat index is 3 GB. PQ compresses to 48 MB. At billion scale, PQ is the only feasible option — 1B × 768 × 4 bytes = 3 TB, but with PQ it is 48 GB."

---

## Experiment 5: nprobe Sweep — Proving nprobe=C Equals Brute Force

**Claim to test:** "Setting nprobe equal to the number of clusters recovers exact brute-force search."

We sweep nprobe from 1 to n_clusters and show recall@10 converging to 1.0.

In [ ]:
# --- Experiment 5: nprobe sweep ---

n = 10000
d = 64
n_clusters = 32
vectors, _ = generate_vectors(n, d, n_clusters=n_clusters)

# Build IVF
ivf = SimpleIVF(n_clusters=n_clusters, nprobe=1)
ivf.build(vectors)

# Generate queries and get ground truth
queries = np.random.randn(50, d)
queries /= np.linalg.norm(queries, axis=1, keepdims=True)
gt = [brute_force_search(q, vectors, k=10) for q in queries]

# Sweep nprobe
nprobe_values = [1, 2, 3, 5, 8, 10, 15, 20, 25, 32]
recall_values = []
latency_values = []

for nprobe in nprobe_values:
    ivf.nprobe = nprobe
    
    t0 = time.perf_counter()
    results = [ivf.search(q, k=10) for q in queries]
    total_time = time.perf_counter() - t0
    latency_values.append(total_time / len(queries) * 1000)  # ms
    
    recalls = []
    for (gt_idx, _), (pred_idx, _) in zip(gt, results):
        if len(pred_idx) > 0:
            recalls.append(recall_at_k(pred_idx, gt_idx, k=10))
    recall_values.append(np.mean(recalls))

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(nprobe_values, recall_values, 'o-', color='#1976d2', linewidth=2, markersize=8)
ax1.set_xlabel('nprobe', fontsize=12)
ax1.set_ylabel('Recall@10', fontsize=12)
ax1.set_title(f'Recall@10 vs nprobe (C={n_clusters})', fontsize=14, fontweight='bold')
ax1.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Perfect recall')
ax1.axvline(x=n_clusters, color='gray', linestyle=':', alpha=0.5, label=f'nprobe=C={n_clusters}')
ax1.legend(fontsize=10)
ax1.set_ylim(0.3, 1.05)
ax1.grid(alpha=0.3)

ax2.plot(nprobe_values, latency_values, 's-', color='#e53935', linewidth=2, markersize=8)
ax2.set_xlabel('nprobe', fontsize=12)
ax2.set_ylabel('Latency (ms per query)', fontsize=12)
ax2.set_title('Latency vs nprobe', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"{'nprobe':>8} {'Recall@10':>10} {'Latency (ms)':>14} {'Clusters searched':>18}")
print("-" * 52)
for np_val, r, l in zip(nprobe_values, recall_values, latency_values):
    pct = np_val / n_clusters * 100
    marker = " ← brute force" if np_val == n_clusters else ""
    print(f"{np_val:>8} {r:>10.3f} {l:>14.3f} {pct:>16.0f}%{marker}")

print(f"\nAt nprobe={n_clusters} (all clusters), recall = {recall_values[-1]:.3f}")
print(f"This confirms: nprobe = n_clusters recovers exact search.")

**What this shows:** Recall increases with nprobe and reaches 1.0 when nprobe equals the total number of clusters (which is equivalent to brute force). The curve is concave — the first few probes gain the most recall. In an interview: "Setting nprobe equal to the number of clusters recovers brute force exactly. The recall curve is concave — nprobe=10 with 100 clusters gives 90%+ recall while searching only 10% of the data."

---

## Summary

| Experiment | Claim Tested | Evidence |
|-----------|-------------|----------|
| 1. BF vs IVF | IVF is faster with small recall cost | 5-50× speedup at 85-95% recall |
| 2. HNSW params | M and ef control the recall-latency trade-off | Pareto frontier shows M=16, ef=50-100 as sweet spot |
| 3. Dimensionality | High dimensions make NN distances converge | NN/avg distance ratio approaches 1.0 as d grows |
| 4. Memory | PQ compresses 60× | Theoretical formulas match actual numpy array sizes |
| 5. nprobe sweep | nprobe=C equals brute force | Recall reaches 1.0 at nprobe=n_clusters |

For the full mathematical treatment, failure modes, and interview Q&A, see [vector-databases-interview.md](./vector-databases-interview.md).

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)